# 🛳️ Train Bi-LSTM Trajectory Model on NOAA AIS Dataset

This notebook trains the **Bi-LSTM Sequence-to-Sequence Model with Bahdanau Attention** directly on `data/AIS_178749379871876535_6390-1787530591097.csv` to accurately predict future vessel trajectories ($T_{\text{in}}=32 \rightarrow T_{\text{out}}=8$).

### Pipeline Overview:
1. **Stream & Filter CSV**: Load NOAA AIS records with valid timestamps and GPS coordinates.
2. **Feature Extraction**: Extract 9 kinematic features per ping (`lat_delta`, `lon_delta`, `sog_norm`, `cog_sin`, `cog_cos`, `time_delta_norm`, `speed_change`, `vessel_type_risk`, `gap_flag`).
3. **Sliding Windows**: Generate sliding sequences for all active vessels.
4. **Train Bi-LSTM**: Train the 2-layer Bidirectional Encoder + Autoregressive Decoder with Cosine Annealing learning rate.
5. **Evaluate & Checkpoint**: Evaluate Haversine distance error (km) and save checkpoint to `ml/results/trajectory_predictor.pt`.
6. **Export Index**: Generate `ml/results/vessel_simulation_index.json` for live frontend animation.

## 1. Imports & Configuration

In [ ]:
import sys
import math
import time
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import yaml
import matplotlib.pyplot as plt

ML_DIR = Path('..').resolve()
sys.path.insert(0, str(ML_DIR))

with open(ML_DIR / 'config.yaml') as f:
    CFG = yaml.safe_load(f)

CSV_PATH = Path('../../data/AIS_178749379871876535_6390-1787530591097.csv')
RESULTS_DIR = ML_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = RESULTS_DIR / 'trajectory_predictor.pt'

DEVICE = (
    torch.device('mps')  if torch.backends.mps.is_available()  else
    torch.device('cuda') if torch.cuda.is_available()            else
    torch.device('cpu')
)

print(f'Using Device: {DEVICE}')
print(f'CSV Path:     {CSV_PATH} (exists={CSV_PATH.exists()})')

## 2. Load NOAA AIS Dataset & Select Top Active Vessels

In [ ]:
from ais_dataset import build_trajectory_features, VESSEL_TYPE_RISK, DEFAULT_RISK

AIS_COLS = {
    'MMSI': 'mmsi', 'BaseDateTime': 'base_date_time',
    'LAT': 'latitude', 'LON': 'longitude',
    'SOG': 'sog', 'COG': 'cog', 'Heading': 'heading',
    'VesselName': 'vessel_name', 'VesselType': 'vessel_type'
}

print('Loading CSV into memory...')
df_raw = pd.read_csv(
    CSV_PATH,
    usecols=list(AIS_COLS.keys()),
    dtype={'MMSI': str, 'VesselName': str, 'VesselType': str},
    parse_dates=['BaseDateTime'],
    low_memory=False
)
df_raw = df_raw.rename(columns=AIS_COLS)
df_raw = df_raw.dropna(subset=['latitude', 'longitude', 'mmsi', 'base_date_time'])
df_raw['sog'] = pd.to_numeric(df_raw['sog'], errors='coerce').fillna(0.0)
df_raw['cog'] = pd.to_numeric(df_raw['cog'], errors='coerce').fillna(0.0)
df_raw['vessel_type'] = pd.to_numeric(df_raw['vessel_type'], errors='coerce').fillna(0).astype(int)

ping_counts = df_raw.groupby('mmsi').size().reset_index(name='ping_count')
ping_counts = ping_counts[ping_counts['ping_count'] >= 50].sort_values('ping_count', ascending=False)
top_mmsis = ping_counts.head(40)['mmsi'].tolist()

print(f'Total records in CSV: {len(df_raw):,}')
print(f'Unique vessels:       {df_raw["mmsi"].nunique():,}')
print(f'Selected top {len(top_mmsis)} vessels for training and trajectory simulation.')

## 3. Extract 9-Dimensional Kinematic Features & Sliding Windows

In [ ]:
T_IN  = 32
T_OUT = 8

trajectories = {}
vessel_meta  = {}

for mmsi in top_mmsis:
    v_df = df_raw[df_raw['mmsi'] == mmsi].sort_values('base_date_time').reset_index(drop=True)
    feats = build_trajectory_features(v_df)
    trajectories[mmsi] = feats
    v_name = str(v_df['vessel_name'].iloc[0]) if 'vessel_name' in v_df.columns and pd.notna(v_df['vessel_name'].iloc[0]) else f'Vessel {mmsi}'
    vessel_meta[mmsi] = {
        'name': v_name,
        'type': int(v_df['vessel_type'].iloc[0]),
        'pings': len(v_df)
    }

class NOAATrajectoryDataset(Dataset):
    def __init__(self, trajectories, t_in=32, t_out=8, stride=4):
        self.windows = []
        win_len = t_in + t_out
        for feats in trajectories.values():
            if len(feats) < win_len: continue
            for start in range(0, len(feats) - win_len + 1, stride):
                src = feats[start : start + t_in]
                tgt = feats[start + t_in : start + win_len, :3]  # lat_delta, lon_delta, sog_norm
                self.windows.append((src, tgt))
                
    def __len__(self):
        return len(self.windows)
        
    def __getitem__(self, idx):
        src, tgt = self.windows[idx]
        return torch.from_numpy(src.copy()), torch.from_numpy(tgt.copy())

# 80/20 train/val split
random.seed(42)
mmsi_list = list(trajectories.keys())
random.shuffle(mmsi_list)
n_train = int(len(mmsi_list) * 0.8)

train_trajs = {m: trajectories[m] for m in mmsi_list[:n_train]}
val_trajs   = {m: trajectories[m] for m in mmsi_list[n_train:]}

train_ds = NOAATrajectoryDataset(train_trajs, t_in=T_IN, t_out=T_OUT, stride=4)
val_ds   = NOAATrajectoryDataset(val_trajs,   t_in=T_IN, t_out=T_OUT, stride=8)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)

print(f'Extracted {len(train_ds):,} training sequences and {len(val_ds):,} validation sequences.')

## 4. Initialize Bi-LSTM Seq2Seq Model & Optimization

In [ ]:
from ais_model import build_ais_model, build_ais_loss

model = build_ais_model(CFG).to(DEVICE)
loss_fn = build_ais_loss(CFG)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

def haversine_km_batch(pred_lat, pred_lon, tgt_lat, tgt_lon):
    R = 6371.0
    dlat = torch.deg2rad(pred_lat - tgt_lat)
    dlon = torch.deg2rad(pred_lon - tgt_lon)
    lat_m = torch.deg2rad((pred_lat + tgt_lat) / 2.0)
    a = dlat**2 + (torch.cos(lat_m) * dlon)**2
    dist = R * torch.sqrt(a.clamp(min=0))
    return dist.mean().item()

print(f'Initialized Bi-LSTM model on {DEVICE} with {sum(p.numel() for p in model.parameters()):,} parameters.')

## 5. Train Model & Track Haversine Errors

In [ ]:
EPOCHS = 15
history = {'train_loss': [], 'val_loss': [], 'val_hav_km': []}
best_val_hav = float('inf')

print(f'Training for {EPOCHS} epochs on NOAA AIS sequences...')
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    teacher_ratio = max(0.0, 0.7 * (1.0 - epoch / (EPOCHS * 0.6)))
    
    # Train phase
    model.train()
    train_loss = 0.0
    for src, tgt in train_loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        pred = model(src, t_out=T_OUT, teacher_input=tgt, teacher_ratio=teacher_ratio)
        loss = loss_fn(pred, tgt)
        
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
        
    train_loss /= len(train_loader)
    scheduler.step()
    
    # Validation phase
    model.eval()
    val_loss, val_hav = 0.0, 0.0
    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            pred = model.predict(src, t_out=T_OUT)
            loss = loss_fn(pred, tgt)
            val_loss += loss.item()
            val_hav += haversine_km_batch(pred[..., 0], pred[..., 1], tgt[..., 0], tgt[..., 1])
            
    val_loss /= len(val_loader)
    val_hav  /= len(val_loader)
    elapsed = time.time() - t0
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_hav_km'].append(val_hav)
    
    is_best = val_hav < best_val_hav
    if is_best:
        best_val_hav = val_hav
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_hav_km': val_hav,
            'val_loss': val_loss,
        }, CKPT_PATH)
        
    print(f'Epoch {epoch:2d}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Error: {val_hav:.3f} km | {elapsed:.1f}s {"⭐ BEST" if is_best else ""}')

print(f'\n✅ Training complete! Best validation error: {best_val_hav:.3f} km')

## 6. Training Loss & Error Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), facecolor='#0b1724')
ax1.set_facecolor('#0b1724')
ax2.set_facecolor('#0b1724')

epochs_arr = range(1, EPOCHS + 1)
ax1.plot(epochs_arr, history['train_loss'], color='#00f2fe', label='Train Loss')
ax1.plot(epochs_arr, history['val_loss'], color='#b877ff', label='Val Loss')
ax1.set_title('Training & Validation Loss', color='#c6f1f7')
ax1.set_xlabel('Epoch', color='#7a9bc0')
ax1.tick_params(colors='#7a9bc0')
ax1.legend(facecolor='#0b1724', labelcolor='white')
for s in ax1.spines.values(): s.set_color('#1e3352')

ax2.plot(epochs_arr, history['val_hav_km'], color='#22c55e', marker='o', label='Haversine Distance (km)')
ax2.set_title('Validation Position Error (km)', color='#c6f1f7')
ax2.set_xlabel('Epoch', color='#7a9bc0')
ax2.tick_params(colors='#7a9bc0')
ax2.legend(facecolor='#0b1724', labelcolor='white')
for s in ax2.spines.values(): s.set_color('#1e3352')

plt.tight_layout()
plt.show()

## 7. Export High-Accuracy Simulation Index for Frontend

In [ ]:
# Load best checkpoint for inference
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return 2 * R * math.asin(math.sqrt(max(0.0, a)))

simulation_results = {}
VESSEL_TYPE_NAMES = {
    range(80, 90): 'Tanker', range(70, 80): 'Cargo',
    range(60, 70): 'Passenger', range(30, 31): 'Fishing', range(50, 60): 'Special Craft / Patrol'
}
def get_type_name(vtype_code):
    for r, name in VESSEL_TYPE_NAMES.items():
        if vtype_code in r: return name
    return 'Other'

for mmsi in top_mmsis:
    v_df = df_raw[df_raw['mmsi'] == mmsi].sort_values('base_date_time').reset_index(drop=True)
    feats = trajectories[mmsi]
    
    min_len = T_IN + T_OUT
    if len(feats) < min_len: continue
    start = max(0, (len(feats) - min_len) // 2)
    src_np = feats[start: start + T_IN]
    
    anchor_lat = float(v_df['latitude'].iloc[start + T_IN - 1])
    anchor_lon = float(v_df['longitude'].iloc[start + T_IN - 1])
    
    actual_lats = v_df['latitude'].iloc[start + T_IN: start + min_len].tolist()
    actual_lons = v_df['longitude'].iloc[start + T_IN: start + min_len].tolist()
    actual_sogs = v_df['sog'].iloc[start + T_IN: start + min_len].tolist()
    
    src_t = torch.from_numpy(src_np.copy()).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred_np = model.predict(src_t, t_out=T_OUT).squeeze(0).cpu().numpy()
        
    pred_lats, pred_lons, pred_sogs = [], [], []
    for i in range(T_OUT):
        pred_lats.append(round(anchor_lat + float(pred_np[i, 0]), 5))
        pred_lons.append(round(anchor_lon + float(pred_np[i, 1]), 5))
        pred_sogs.append(round(float(pred_np[i, 2]) * 30.0, 2))
        
    step_errors = []
    for i in range(T_OUT):
        err = haversine_km(actual_lats[i], actual_lons[i], pred_lats[i], pred_lons[i])
        step_errors.append(round(err, 3))
        
    hist_lats = v_df['latitude'].iloc[start: start + T_IN].tolist()
    hist_lons = v_df['longitude'].iloc[start: start + T_IN].tolist()
    vtype_code = int(v_df['vessel_type'].iloc[0])
    vname = str(v_df['vessel_name'].iloc[0]) if 'vessel_name' in v_df.columns and pd.notna(v_df['vessel_name'].iloc[0]) else f'Vessel {mmsi}'
    
    simulation_results[mmsi] = {
        'mmsi': mmsi,
        'vessel_name': vname,
        'vessel_type': get_type_name(vtype_code),
        'risk_weight': VESSEL_TYPE_RISK.get(vtype_code, DEFAULT_RISK),
        'ping_count': len(v_df),
        'history_track': [[lon, lat] for lat, lon in zip(hist_lats, hist_lons)],
        'actual_track': [[lon, lat] for lat, lon in zip(actual_lats, actual_lons)],
        'predicted_track': [[lon, lat] for lat, lon in zip(pred_lats, pred_lons)],
        'actual_sogs': [round(s, 1) for s in actual_sogs],
        'predicted_sogs': pred_sogs,
        'step_errors_km': step_errors,
        'mean_error_km': round(float(np.mean(step_errors)), 3),
        'anchor_lat': anchor_lat,
        'anchor_lon': anchor_lon,
    }

sim_out_path = RESULTS_DIR / 'vessel_simulation_index.json'
with open(sim_out_path, 'w') as f:
    json.dump({
        'generated_at': pd.Timestamp.now().isoformat(),
        'csv_source': CSV_PATH.name,
        'model_checkpoint': str(CKPT_PATH.name),
        'model_metrics': {
            'trained_mean_error_km': round(float(np.mean([r['mean_error_km'] for r in simulation_results.values()])), 3)
        },
        't_in': T_IN, 't_out': T_OUT,
        'vessels': list(simulation_results.values())
    }, f, indent=2)

print(f'✅ Saved simulation index with {len(simulation_results)} vessels to {sim_out_path}')
print(f'   Average Trajectory Error: {np.mean([r["mean_error_km"] for r in simulation_results.values()]):.3f} km')